# Симуляция бэкенд-пайплайна — KEYWORDS (без embeddings)

Старый подход: проверка по ключевым словам

In [ ]:
# ============================================================
# 1. Загрузка модели
# ============================================================
import joblib
import numpy as np
import pandas as pd
import re
from pathlib import Path

ML_FEATURES = [
    "gross_output_growth_yoy", "land_to_livestock_ratio",
    "historical_survival_rate", "subsidy_dependence_index",
    "veterinary_compliance", "years_in_operation",
    "pedigree_ratio", "previous_subsidies_count",
    "debt_load_ratio", "log_amount", "livestock_count",
    "direction_code", "is_pedigree", "is_producer",
    "hour_submitted", "month_submitted", "region_encoded",
]

model = joblib.load("../models/xgb_scorer.joblib")
scaler = joblib.load("../models/scaler.joblib")

print("Модель загружена")

In [ ]:
# ============================================================
# 2. Форма + сканирование PDF
# ============================================================

print("=" * 60)
print("  ФОРМА ЗАЯВКИ НА СУБСИДИЮ")
print("=" * 60)

form = {
    "company_name":    "ТОО Агро-Эталон",
    "bin_iin":         "050140001111",
    "region":          "Алматинская область",
    "direction":       "Мясное",
    "subsidy_type":    "Субсидирование племенного КРС",
    "requested_amount": 26000000,
}

optional = {
    "farm_size":       None,
    "debt_level":      None,
    "subsidy_exp":     None,
    "vet_status":      None,
    "growth_choice":   None,
    "pedigree_choice": None,
}

pdf_folder = "/home/haku/pdf/"

print(f"Компания: {form['company_name']}")
print(f"БИН: {form['bin_iin']}")
print(f"Регион: {form['region']}")
print(f"Направление: {form['direction']}")
print(f"Сумма: {form['requested_amount']:,.0f} тенге")

_folder = Path(pdf_folder)
pdf_files = []
if _folder.exists():
    pdf_files = sorted(_folder.glob("*.pdf"))
    if pdf_files:
        print(f"\nНайдено PDF файлов: {len(pdf_files)}")
        for i, f in enumerate(pdf_files, 1):
            print(f"  {i}. {f.name} ({f.stat().st_size / 1024:.0f} КБ)")
    else:
        print(f"\n⚠️ В папке нет PDF файлов")
else:
    print(f"\n⚠️ Папка {pdf_folder} не существует")

In [ ]:
# ============================================================
# 3. Заполняем 17 фичей
# ============================================================

DEFAULTS = {
    "gross_output_growth_yoy":  0.05,
    "land_to_livestock_ratio":  2.0,
    "historical_survival_rate": 0.87,
    "subsidy_dependence_index": 0.30,
    "veterinary_compliance":    0.85,
    "years_in_operation":       5.0,
    "pedigree_ratio":           0.50,
    "previous_subsidies_count": 3.0,
    "debt_load_ratio":          1.5,
    "log_amount":               15.0,
    "livestock_count":          10.0,
    "direction_code":           0.0,
    "is_pedigree":              0.0,
    "is_producer":              0.0,
    "hour_submitted":           12.0,
    "month_submitted":          1.0,
    "region_encoded":           0.0,
}

DIRECTION_MAP = {"Мясное": 0, "Молочное": 0, "Овцеводство": 1, "Птицеводство": 3, "Коневодство": 2, "Верблюдоводство": 4, "Свиноводство": 5}
REGION_MAP = {"Алматинская область": 0, "Акмолинская область": 1, "Атырауская область": 2, "Мангистауская область": 3, "Жамбылская область": 4}

features = DEFAULTS.copy()
features["log_amount"] = np.log1p(form["requested_amount"])
features["livestock_count"] = form["requested_amount"] / 260000
features["direction_code"] = DIRECTION_MAP.get(form["direction"], 0)
features["region_encoded"] = REGION_MAP.get(form["region"], 0)

if "племен" in form["subsidy_type"].lower():
    features["is_pedigree"] = 1.0
if "производит" in form["subsidy_type"].lower():
    features["is_producer"] = 1.0

if optional.get("debt_level") == "Низкая":
    features["debt_load_ratio"] = 0.8
elif optional.get("debt_level") == "Высокая":
    features["debt_load_ratio"] = 3.5
if optional.get("growth_choice") == "Рост":
    features["gross_output_growth_yoy"] = 0.20
elif optional.get("growth_choice") == "Спад":
    features["gross_output_growth_yoy"] = -0.10
if optional.get("pedigree_choice") == ">60%":
    features["pedigree_ratio"] = 0.75
elif optional.get("pedigree_choice") == "<20%":
    features["pedigree_ratio"] = 0.15
if optional.get("vet_status") == "Нарушения":
    features["veterinary_compliance"] = 0.40

print("Фичи для модели:")
for k, v in features.items():
    print(f"  {k}: {v}")

In [ ]:
# ============================================================
# 4. Извлечение текста из PDF + ВЫВОД ТЕКСТА
# ============================================================

combined_text = ""
extraction_note = ""

print(f"Обрабатываем PDF файлов: {len(pdf_files)}")

for pdf_path in pdf_files:
    print(f"\n{'='*60}")
    print(f"  ФАЙЛ: {pdf_path.name}")
    print(f"{'='*60}")
    
    try:
        from pypdf import PdfReader
        reader = PdfReader(str(pdf_path))
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
        combined_text += text + "\n\n"
        
        # Выводим текст чтобы видеть что анализируется
        print(f"  Извлечено: {len(text)} симв.\n")
        print(f"  --- ТЕКСТ ---")
        print(text[:2000])  # первые 2000 символов
        if len(text) > 2000:
            print(f"  ... (ещё {len(text)-2000} симв.)")
        print(f"  --- КОНЕЦ ТЕКСТА ---")
    except Exception as e:
        extraction_note += f"Ошибка {pdf_path.name}: {e}\n"
        print(f"  ОШИБКА: {e}")

if not combined_text:
    print("\nТекст не извлечён ни из одного файла")
else:
    print(f"\n{'='*60}")
    print(f"  ВСЕГО текста из {len(pdf_files)} файлов: {len(combined_text)} симв.")
    print(f"{'='*60}")

In [ ]:
# ============================================================
# 5. ML Score
# ============================================================

X = pd.DataFrame([features], columns=ML_FEATURES)
X_scaled = scaler.transform(X)
base_score = float(np.clip(model.predict(X_scaled)[0], 1, 100))

print(f"ML Score: {base_score:.1f}")

In [ ]:
# ============================================================
# 6. Compliance Check — KEYWORDS (старый подход)
# ============================================================

import sys
sys.path.insert(0, str(Path("../").resolve()))
from ml.compliance_checker import SUBSIDY_RULES, UNIVERSAL_DOCUMENT_CHECKLIST

def detect_subsidy_type(subsidy_name):
    t = subsidy_name.lower()
    if "бык" in t: return "КРС_быки"
    if "молок" in t: return "КРС_молоко"
    if "баран" in t: return "овцы_бараны"
    return "КРС_маточное"

def check_compliance_keywords(doc_text, rules):
    """Старый keyword-подход как в compliance_checker.py"""
    text_lower = doc_text.lower()
    results = []
    
    # Основные требования
    for req in rules["requirements"]:
        found_kw = [kw for kw in req["keywords"] if kw.lower() in text_lower]
        
        if len(found_kw) >= 2:
            status = "ВЫПОЛНЕНО"
            evidence = f"Найдены ключевые слова: {', '.join(found_kw[:3])}"
        elif len(found_kw) == 1:
            status = "ЧАСТИЧНО"
            evidence = f"Частично: '{found_kw[0]}'"
        else:
            status = "НЕ НАЙДЕНО"
            evidence = f"Не найдено: {', '.join(req['keywords'][:3])}"
        
        results.append({
            "id": req["id"], "text": req["text"], "status": status,
            "evidence": evidence, "critical": req["critical"],
            "found_kw": found_kw, "kw_count": len(found_kw)
        })
    
    # Универсальные требования
    for req in UNIVERSAL_DOCUMENT_CHECKLIST:
        found_kw = [kw for kw in req["keywords"] if kw.lower() in text_lower]
        status = "ВЫПОЛНЕНО" if len(found_kw) >= 1 else "НЕ НАЙДЕНО"
        evidence = f"Найдено: {', '.join(found_kw)}" if found_kw else "Не найдено"
        results.append({
            "id": req["id"], "text": req["text"], "status": status,
            "evidence": evidence, "critical": req["critical"],
            "found_kw": found_kw, "kw_count": len(found_kw)
        })
    
    return results

if combined_text.strip():
    subsidy_key = detect_subsidy_type(form["subsidy_type"])
    rules = SUBSIDY_RULES[subsidy_key]
    compliance_results = check_compliance_keywords(combined_text, rules)
    
    done = sum(1 for r in compliance_results if r["status"] == "ВЫПОЛНЕНО")
    partial = sum(1 for r in compliance_results if r["status"] == "ЧАСТИЧНО")
    total = len(compliance_results)
    doc_score = (done + partial * 0.5) / total * 100
    doc_completeness = min(1.0, len(combined_text) / 8000)
    
    print(f"\n{'='*60}")
    print(f"  COMPLIANCE CHECK (KEYWORDS)")
    print(f"{'='*60}")
    for r in compliance_results:
        if r["status"] == "ВЫПОЛНЕНО":
            emoji = "✅"
        elif r["status"] == "ЧАСТИЧНО":
            emoji = "⚠️"
        else:
            emoji = "❌"
        crit = " [КРИТ]" if r["critical"] else ""
        print(f"  {emoji} {r['id']}: {r['status']}{crit}")
        print(f"     {r['evidence']}")
    
    print(f"\nDoc Score: {doc_score:.0f}% ({done} выполнено, {partial} частично из {total})")
    print(f"Doc Completeness: {doc_completeness:.2f}")
else:
    doc_score = None
    doc_completeness = 0.0
    compliance_results = []
    print("\nНет документов — Compliance Check пропущен")

In [ ]:
# ============================================================
# 7. Финальная формула (документы важнее ML)
# ============================================================

if doc_score is not None:
    if doc_completeness >= 0.70:
        ml_w, doc_w = 0.30, 0.70
    elif doc_completeness >= 0.40:
        ml_w, doc_w = 0.50, 0.50
    else:
        ml_w, doc_w = 0.70, 0.30
    
    final_score = ml_w * base_score + doc_w * doc_score
    final_score = round(max(1.0, min(100.0, final_score)), 1)
else:
    ml_w, doc_w = 1.0, 0.0
    final_score = base_score

if final_score >= 80:
    zone = "green"
    rec = "Строго рекомендовано"
elif final_score >= 50:
    zone = "yellow"
    rec = "Требует рассмотрения"
else:
    zone = "red"
    rec = "Не рекомендовано"

print("\n" + "=" * 60)
print(f"  РЕЗУЛЬТАТ")
print("=" * 60)
print(f"  ML Score:       {base_score:.1f}")
print(f"  Doc Score:      {doc_score:.0f}" if doc_score else "  Doc Score:      нет документов")
print(f"  Веса:           ML {ml_w:.0%} / Doc {doc_w:.0%}")
print(f"  FINAL SCORE:    {final_score:.1f} / 100")
print(f"  Зона:           {zone.upper()}")
print(f"  Рекомендация:   {rec}")
print("=" * 60)

In [ ]:
# ============================================================
# 8. SHAP объяснение
# ============================================================
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_scaled)
if shap_values.ndim == 2:
    shap_values = shap_values[0]

FEATURE_LABELS = {
    "gross_output_growth_yoy": "Рост продукции",
    "land_to_livestock_ratio": "Пастбища",
    "historical_survival_rate": "Сохранность",
    "subsidy_dependence_index": "Зависимость от субсидий",
    "veterinary_compliance": "Ветеринария",
    "years_in_operation": "Стаж",
    "pedigree_ratio": "Племенное поголовье",
    "previous_subsidies_count": "История субсидий",
    "debt_load_ratio": "Долговая нагрузка",
    "log_amount": "Масштаб заявки",
    "livestock_count": "Поголовье",
    "direction_code": "Направление",
    "is_pedigree": "Племенная субсидия",
    "is_producer": "Производители",
    "hour_submitted": "Час подачи",
    "month_submitted": "Месяц подачи",
    "region_encoded": "Регион",
}

factors = []
for name, sv in zip(ML_FEATURES, shap_values):
    factors.append({"label": FEATURE_LABELS.get(name, name), "value": features[name], "shap": sv})

factors.sort(key=lambda x: abs(x["shap"]), reverse=True)
pos = [f for f in factors if f["shap"] > 0][:3]
neg = [f for f in factors if f["shap"] < 0][:3]

print("\nSHAP — топ влияния:")
if pos:
    print("  Повышают:")
    for f in pos:
        print(f"    +{f['shap']:.2f}: {f['label']} = {f['value']:.2f}")
if neg:
    print("  Снижают:")
    for f in neg:
        print(f"    {f['shap']:.2f}: {f['label']} = {f['value']:.2f}")